# Multivariate Segmentation with BayesBreak

This tutorial demonstrates how to segment **multivariate (vector-valued) time series** using `BayesBreakMultivariate`. This is useful when you have multiple channels that share the same underlying change-point structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bayesbreak import BayesBreakGaussian, BayesBreakMultivariate

rng = np.random.default_rng(123)
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Generate Multi-Channel Data

We create a 3-channel signal where all channels share the same boundary locations but have different segment means.

In [ ]:
# Shared segmentation structure
segment_lengths = [50, 70, 80]
n_total = sum(segment_lengths)

# Channel-specific means for each segment
channel_means = [
    [0.0, 2.0, -1.0],   # Channel 0
    [1.0, -1.0, 3.0],   # Channel 1
    [2.0, 0.0, 1.0],    # Channel 2
]

# Generate data
n_channels = len(channel_means)
Y = np.zeros((n_total, n_channels))

for ch, means in enumerate(channel_means):
    signal = np.concatenate([np.full(n, mu) for mu, n in zip(means, segment_lengths)])
    Y[:, ch] = signal + 0.3 * rng.standard_normal(n_total)

print(f"Data shape: {Y.shape}")
print(f"True boundaries: {np.cumsum(segment_lengths[:-1]).tolist()}")

In [ ]:
# Visualize all channels
fig, axes = plt.subplots(n_channels, 1, figsize=(12, 8), sharex=True)
colors = ['steelblue', 'coral', 'seagreen']

for ch, ax in enumerate(axes):
    ax.plot(Y[:, ch], 'o', alpha=0.5, markersize=2, color=colors[ch])
    for b in np.cumsum(segment_lengths[:-1]):
        ax.axvline(b, color='red', linestyle='--', alpha=0.6)
    ax.set_ylabel(f'Channel {ch}')

axes[-1].set_xlabel('Index')
axes[0].set_title('Multi-Channel Data with Shared Boundaries')
plt.tight_layout()
plt.show()

## 2. Fit Multivariate Model

`BayesBreakMultivariate` wraps a base estimator and performs shared-boundary segmentation across channels.

In [ ]:
# Create multivariate model
base_estimator = BayesBreakGaussian(k_max=10)
mv_model = BayesBreakMultivariate(base_estimator)

# Fit to multivariate data
mv_model.fit(Y)

print(f"Estimated segments: {mv_model.get_segment_count()}")
print(f"Estimated boundaries: {mv_model.get_boundaries()}")

In [ ]:
# Get predictions for all channels
Y_pred = mv_model.predict()

# Visualize results
fig, axes = plt.subplots(n_channels, 1, figsize=(12, 8), sharex=True)

for ch, ax in enumerate(axes):
    ax.plot(Y[:, ch], 'o', alpha=0.3, markersize=2, color='gray')
    ax.plot(Y_pred[:, ch], '-', linewidth=2, color=colors[ch], label='Fit')
    for b in mv_model.get_boundaries():
        ax.axvline(b, color='blue', linestyle=':', alpha=0.5)
    ax.set_ylabel(f'Channel {ch}')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Index')
axes[0].set_title('Multivariate Segmentation Results')
plt.tight_layout()
plt.show()

## 3. Comparing Univariate vs Multivariate

Let's see how joint modeling improves boundary detection compared to fitting each channel independently.

In [ ]:
# Fit each channel independently
univariate_results = []
for ch in range(n_channels):
    model = BayesBreakGaussian(k_max=10)
    model.fit(Y[:, ch])
    univariate_results.append({
        'k': model.get_segment_count(),
        'boundaries': model.get_boundaries()
    })

print("Independent univariate fits:")
for ch, res in enumerate(univariate_results):
    print(f"  Channel {ch}: k={res['k']}, boundaries={res['boundaries']}")

print(f"\nMultivariate (joint) fit:")
print(f"  k={mv_model.get_segment_count()}, boundaries={mv_model.get_boundaries()}")
print(f"\nTrue boundaries: {np.cumsum(segment_lengths[:-1]).tolist()}")

## 4. Summary

`BayesBreakMultivariate` is useful when:
- Multiple channels share the same underlying segmentation structure
- Individual channels may be noisy but collectively provide stronger evidence
- You want a single coherent segmentation across all channels

The model assumes **independent observations across channels** given the segment boundaries, with channel-specific parameters within each segment.